# Reinforcement Learning from Verifiable Rewards (RLVR) with SageMaker

## Lab 2 — Fine-Tuning with RLVRTrainer

In this lab, you will launch a **serverless RLVR training job** using the SageMaker `RLVRTrainer` to fine-tune **Qwen 3 — 0.6B Instruct** on the math dataset you prepared in Lab 1.

### What you'll do in this notebook

1. Set up your SageMaker session and retrieve the datasets from Lab 1
2. Create a Model Package Group to store the fine-tuned model
3. Configure the `RLVRTrainer` with model, data, and hyperparameters
4. Submit the serverless training job and monitor progress

### How serverless RLVR training works

With SageMaker serverless customization, you don't need to provision or manage any compute infrastructure. SageMaker automatically:
- Selects the right instance types and cluster size for your model
- Runs the RLVR training loop (policy rollouts → reward verification → policy update)
- Logs metrics to MLflow for experiment tracking
- Saves the fine-tuned model to S3 and registers it in the Model Registry

---

## 1. Prerequisites

### Set up the SageMaker session

We initialize the SageMaker session and resolve the IAM role, just like in Lab 1.

In [ ]:
import boto3
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Prepare experiment tracking

Reuse a ready MLflow App before configuring the trainer. An optional
`MLFLOW_APP_ARN` in `config.py` wins; otherwise the shared resolver discovers the
tagged stack app before the named-app/native SDK fallback. Ambiguity, access
errors, and failed or timed-out apps stop setup instead of starting untracked jobs.


In [ ]:
from config import resolve_mlflow_app

mlflow_app_arn = resolve_mlflow_app(sm_client, session=sess)


### Retrieve datasets and configure the job

We fetch the training and validation datasets that were registered in the SageMaker AI Registry during Lab 1. We also define the base model ID, output path, and MLflow experiment name.

In [ ]:
response = sm_client.list_hub_contents(
    HubName='SageMakerPublicHub',
    HubContentType='Model',
    NameContains='qwen3'
)
for item in response['HubContentSummaries']:
    print(item['HubContentName'])

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

# Required config
# base_model_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_id = "huggingface-reasoning-qwen3-06b"
training_dataset = DataSet.get(name="rlvr-train")
validation_dataset = DataSet.get(name="rlvr-val")

# Optional configs
# SageMaker appends a "-YYYYMMDDHHMMSS" timestamp then truncates the job name to
# 63 chars; keep base_job_name short enough that the timestamp always survives so
# each run gets a unique name (avoids ResourceInUse on re-runs / longer models).
job_name = f"rlvr-{base_model_id.split('/')[-1].replace('.', '-')}"[: 63 - 15].rstrip("-")
mlflow_experiment_name = "qwen3-06b-rlvr"

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}-rlvr"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}-rlvr"

os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = (
    f"https://mlflow.sagemaker.{sess.boto_region_name}.app.aws"
)

print(f"Base model:    {base_model_id}")
print(f"Training data: {training_dataset.arn}")
print(f"Val data:      {validation_dataset.arn}")
print(f"Output path:   {output_path}")

---

## 2. Create a Model Package Group

A **Model Package Group** is a container in the SageMaker Model Registry that holds versioned model artifacts. Each time you run a training job, the resulting model is saved as a new version in this group — making it easy to compare runs and roll back if needed.

In [ ]:
import hashlib
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

MAX_MPG_NAME_LENGTH = 63
suffix = "-rlvr"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Store models from SageMaker serverless RLVR customization",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

---

## 3. Configure the RLVR Trainer

The `RLVRTrainer` is the high-level API for serverless RLVR training. You provide:

| Parameter | Description |
|---|---|
| `model` | Base model ID from the SageMaker Hub |
| `training_dataset` | Registered training dataset from the AI Registry |
| `validation_dataset` | Registered validation dataset from the AI Registry |
| `model_package_group` | Where to store the fine-tuned model versions |
| `s3_output_path` | S3 location for model artifacts |
| `mlflow_experiment_name` | MLflow experiment name for metric tracking |

> **Note:** Setting `accept_eula=True` is required for gated models.

#### Where is the reward function?

There is no `custom_reward_function` argument here. This lab uses the **built-in
rule-based verifier**, which does exact-match checking against the ground truth. It is
selected per record by the `reward_model` field the previous notebook wrote:

```json
"reward_model": {"ground_truth": "72", "style": "rule"}
```

`style: "rule"` is what routes each sample to the built-in verifier, so the reward is
declared in the dataset rather than passed to the trainer. Exact match works here
because GSM8K answers are single numbers, which are either right or wrong with nothing
to judge.

For tasks whose answers cannot be compared that way, you supply your own reward
function instead, which is what the next lab does with a Lambda-backed evaluator.

In [ ]:
from sagemaker.train.rlvr_trainer import RLVRTrainer

trainer = RLVRTrainer(
    mlflow_resource_arn=mlflow_app_arn,
    model=base_model_id,
    model_package_group=model_package_group,
    training_dataset=training_dataset,
    validation_dataset=validation_dataset,
    s3_output_path=output_path,
    mlflow_experiment_name=mlflow_experiment_name,
    base_job_name=job_name,
    sagemaker_session=sess,
    accept_eula=True,
    role=role
)

### View hyperparameters

The trainer comes with sensible defaults. Let's inspect them first, you can also override the ones you want to change.

In [ ]:
from rich.pretty import pprint

print("Default finetuning options:")
pprint(trainer.hyperparameters.to_dict())

In [ ]:
trainer.hyperparameters.max_epochs = 1
trainer.hyperparameters.learning_rate = 5e-06
trainer.hyperparameters.global_batch_size = 128
trainer.hyperparameters.max_prompt_length = 1024

---

## 4. Submit the training job

Calling `trainer.train(wait=False)` submits the job and returns immediately without blocking the notebook, so you can monitor progress separately. SageMaker will:

1. Provision the compute cluster automatically
2. Download the base model and datasets
3. Run the RLVR training loop for the configured number of epochs
4. Upload the fine-tuned model artifacts to S3
5. Register the model in the Model Package Group

> **⏱ Expected duration:** 15-20 minutes. You can monitor progress in SageMaker Studio under **Jobs → Training**, or check the MLflow experiment for live metrics.

In [ ]:
from rich.pretty import pprint

training_job = trainer.train(wait=False)

TRAINING_JOB_NAME = training_job.training_job_name

pprint(training_job)

## Monitor this run

Open the MLflow app and select the experiment printed below. Metrics can appear only
after training starts. The app link expires after five minutes; rerun this link cell
for a fresh URL. Keep the exact training job name when reconnecting to the run.


In [ ]:
TRAINING_JOB_NAME = training_job.training_job_name
region = sess.boto_region_name
console = f"https://{region}.console.aws.amazon.com"
print(f"Job: {console}/sagemaker/home?region={region}#/jobs/{TRAINING_JOB_NAME}")
print(f"Logs: {console}/cloudwatch/home?region={region}#logsV2:log-groups/log-group/"
      f"$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3D{TRAINING_JOB_NAME}")
url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_app_arn, ExpiresInSeconds=300)["AuthorizedUrl"]
print(f"MLflow (open within 5 minutes): {url}")
print(f"Select experiment: {mlflow_experiment_name}")


---

## Next steps

Your RLVR-tuned model is now registered in the SageMaker Model Registry. In the next notebook, you'll evaluate it against the base model to measure the improvement on math reasoning tasks.